## Extract Frames

In [5]:
import os
import argparse
import cv2
import pandas as pd
import subprocess
from datetime import datetime, timedelta
from glob import glob
import piexif
import json
import concurrent.futures
from tqdm import tqdm  # Regular tqdm instead of notebook version
import numpy as np
import time
from functools import partial

In [8]:
def extract_frames_optimized(video_path, output_folder, frame_interval, master_csv_path):
    """
    Optimized version of extract_frames that uses more efficient frame seeking
    and better I/O handling
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get video file information
    video_filename = os.path.basename(video_path)
    video_name = os.path.splitext(video_filename)[0]
    
    # Skip macOS metadata files
    if video_filename.startswith('._'):
        print(f"Skipping macOS metadata file: {video_path}")
        return pd.DataFrame(), False
    
    # Check if this video has already been processed in the master CSV
    if os.path.exists(master_csv_path) and os.path.getsize(master_csv_path) > 0:
        try:
            existing_df = pd.read_csv(master_csv_path)
            # Add check for source_video_path which is more unique than just video_name
            if ('source_video_path' in existing_df.columns and 
                video_path in existing_df['source_video_path'].values):
                return pd.DataFrame(), False
        except Exception as e:
            # If we can't read the CSV, reinitialize it
            print(f"Warning: Error reading master CSV: {str(e)}. Reinitializing CSV.")
            columns = [
                'filename', 'filepath', 'folder_name', 'relative_folder_path', 
                'frame_number', 'frame_time_seconds', 'frame_time_hhmmss', 
                'video_name', 'video_fps', 'video_duration', 'source_video_path'
            ]
            pd.DataFrame(columns=columns).to_csv(master_csv_path, index=False)
    
    # Open the video file
    video = cv2.VideoCapture(video_path)
    if not video.isOpened():
        print(f"Could not open video file: {video_path}")
        return pd.DataFrame(), False
    
    # Get video properties
    fps = video.get(cv2.CAP_PROP_FPS)
    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    
    # Calculate which frames to extract
    frames_to_extract = list(range(0, frame_count, frame_interval))
    
    # Process video frames more efficiently
    frame_data = []
    
    # Set JPEG compression parameters for faster writing
    encode_params = [int(cv2.IMWRITE_JPEG_QUALITY), 95]
    
    for frame_idx in frames_to_extract:
        # Seek to the specific frame directly instead of reading all frames
        video.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame = video.read()
        
        if not success:
            print(f"Failed to read frame {frame_idx} from {video_path}")
            continue
        
        # Calculate time in seconds
        frame_time_seconds = frame_idx / fps
        
        # Format time as HH:MM:SS.ms
        time_obj = timedelta(seconds=frame_time_seconds)
        hours, remainder = divmod(time_obj.seconds, 3600)
        minutes, seconds = divmod(remainder, 60)
        milliseconds = int(time_obj.microseconds / 1000)
        
        # Format time string for filename (HHMMSS_ms)
        time_str_filename = f"{hours:02d}{minutes:02d}{seconds:02d}_{milliseconds:03d}"
        
        # Format time string for display (HH:MM:SS.ms)
        time_str_display = f"{hours:02d}:{minutes:02d}:{seconds:02d}.{milliseconds:03d}"
        
        # Format frame number with leading zeros
        frame_num = str(frame_idx).zfill(len(str(frame_count)))
        
        # Create filename with time information
        filename = f"{video_name}_frame{frame_num}_t{time_str_filename}.jpg"
        filepath = os.path.join(output_folder, filename)
        
        # Save the frame with optimized parameters
        cv2.imwrite(filepath, frame, encode_params)
        
        # Calculate relative folder path
        relative_folder_path = os.path.relpath(output_folder, os.path.dirname(master_csv_path))
        
        # Store frame info - use source_video_path for uniqueness instead of creating a new field
        frame_info = {
            'filename': filename,
            'filepath': filepath,
            'folder_name': os.path.basename(output_folder),
            'relative_folder_path': relative_folder_path,
            'frame_number': frame_idx,
            'frame_time_seconds': frame_time_seconds,
            'frame_time_hhmmss': time_str_display,
            'video_name': video_name,
            'video_fps': fps,
            'video_duration': duration,
            'source_video_path': video_path
        }
        
        # Add frame info to our list
        frame_data.append(frame_info)
    
    if len(frame_data) == 0:
        print(f"Warning: No frames extracted from {video_path}. Video may be corrupt or unreadable.")

    video.release()
    
    # Create DataFrame from frame data
    new_frame_df = pd.DataFrame(frame_data)
    
    # Update master CSV with simplified approach
    if not new_frame_df.empty:
        # Use a simple lock file
        lock_file = f"{master_csv_path}.lock"
        
        # Try to acquire the lock
        lock_acquired = False
        max_attempts = 10
        for attempt in range(max_attempts):
            if not os.path.exists(lock_file):
                # Create lock file
                with open(lock_file, 'w') as f:
                    f.write(str(os.getpid()))
                lock_acquired = True
                break
            else:
                # Wait and retry
                time.sleep(0.2 * (attempt + 1))  # Exponential backoff
        
        if lock_acquired:
            try:
                # Update the CSV
                if os.path.exists(master_csv_path) and os.path.getsize(master_csv_path) > 0:
                    try:
                        existing_df = pd.read_csv(master_csv_path)
                        updated_df = pd.concat([existing_df, new_frame_df], ignore_index=True)
                        updated_df.to_csv(master_csv_path, index=False)
                    except Exception:
                        # If reading fails, just write the new frame data
                        new_frame_df.to_csv(master_csv_path, index=False)
                else:
                    # Master CSV doesn't exist or is empty, create it
                    new_frame_df.to_csv(master_csv_path, index=False)
            except Exception as e:
                print(f"Error updating CSV: {str(e)}")
            finally:
                # Always release the lock
                if os.path.exists(lock_file):
                    os.remove(lock_file)
        else:
            print(f"Could not acquire lock for {video_path}. Proceeding without updating master CSV.")
    
    return new_frame_df, True

def process_video_folder_parallel(input_folder, output_folder, frame_interval, master_csv_path, 
                                  video_extensions=['.mp4', '.avi', '.mov', '.mkv'], 
                                  max_workers=None):
    """
    Process all videos in a folder using parallel processing
    """
    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)
    
    # Ensure the directory for the master CSV exists
    master_csv_dir = os.path.dirname(master_csv_path)
    os.makedirs(master_csv_dir, exist_ok=True)
    
    # Create master CSV with proper columns if it doesn't exist or is empty
    if not os.path.exists(master_csv_path) or os.path.getsize(master_csv_path) == 0:
        # Define the columns that will be in our DataFrame
        columns = [
            'filename', 'filepath', 'folder_name', 'relative_folder_path', 
            'frame_number', 'frame_time_seconds', 'frame_time_hhmmss', 
            'video_name', 'video_fps', 'video_duration', 'source_video_path'
        ]
        
        # Create empty DataFrame with these columns and save it
        empty_df = pd.DataFrame(columns=columns)
        empty_df.to_csv(master_csv_path, index=False)
        print(f"Created new master CSV with proper columns at: {master_csv_path}")
    
    all_files = []
    
    for root, _, files in os.walk(input_folder):
        for file in files:
            if any(file.lower().endswith(ext.lower()) for ext in video_extensions):
                all_files.append(os.path.join(root, file))
    
    all_files.sort()
    
    print(f"Found {len(all_files)} video files")
    
    # Prepare args for each file
    file_infos = []
    for video_path in all_files:
        # Get relative path to maintain folder structure
        rel_path = os.path.relpath(video_path, input_folder)
        rel_dir = os.path.dirname(rel_path)
        
        # Create output directory that mirrors the input structure
        video_output_folder = os.path.join(output_folder, rel_dir)
        os.makedirs(video_output_folder, exist_ok=True)
        
        file_infos.append((video_path, video_output_folder))
    
    # Process videos in parallel
    processed_count = 0
    skipped_count = 0
    
    # Create a partial function with fixed parameters
    process_func = partial(
        _process_single_video, 
        frame_interval=frame_interval, 
        master_csv_path=master_csv_path
    )
    
    # Use ThreadPoolExecutor for I/O bound operations
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks and get future objects
        future_to_file = {
            executor.submit(process_func, video_path, output_dir): (video_path, i) 
            for i, (video_path, output_dir) in enumerate(file_infos)
        }
        
        # Process results as they complete
        for future in tqdm(concurrent.futures.as_completed(future_to_file), 
                          total=len(file_infos), 
                          desc="Processing Videos"):
            video_path, idx = future_to_file[future]
            try:
                was_processed = future.result()
                if was_processed:
                    processed_count += 1
                else:
                    skipped_count += 1
            except Exception as e:
                print(f"Error processing {video_path}: {str(e)}")
    
    print(f"\nAll done. Processed {processed_count} videos, skipped {skipped_count} videos that were already processed.")
    return processed_count, skipped_count

def _process_single_video(video_path, output_dir, frame_interval, master_csv_path):
    """Helper function for parallel processing"""
    try:
        _, was_processed = extract_frames_optimized(
            video_path, 
            output_dir, 
            frame_interval,
            master_csv_path
        )
        return was_processed
    except Exception as e:
        print(f"Error processing {video_path}: {str(e)}")
        return False

In [9]:
start_time = time.time()
    
processed, skipped = process_video_folder_parallel(
        "../mumbai_vids/", 
        "../mumbai_video_frames", 
        1000, 
        "../mumbai_frame_extraction_log.csv", 
        video_extensions=['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv'],
        max_workers= 4
    )
    
elapsed = time.time() - start_time
print(f"Completed in {elapsed:.2f} seconds. Processed: {processed}, Skipped: {skipped}")

Found 76 video files
Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_1_2.MP4
Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_1_1.MP4
Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_2_2.MP4
Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_3.MP4
Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_4.MP4
Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_2_3.MP4
Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_5.MP4
Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_2_1.MP4


Processing Videos: 100%|█████████████████████████████████████████████████████████| 76/76 [00:00<00:00, 516.25it/s]

Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_8_2.MP4Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_6.MP4
Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_7.MP4

Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_8_1.MP4
Skipping macOS metadata file: ../mumbai_vids/day1_3_27/._itinerary_9.MP4
Skipping macOS metadata file: ../mumbai_vids/day2_3_29/._itinerary_1_2.MP4
Skipping macOS metadata file: ../mumbai_vids/day2_3_29/._dadar_flower_market.MP4
Skipping macOS metadata file: ../mumbai_vids/day2_3_29/._itinerary_2_2.MP4
Skipping macOS metadata file: ../mumbai_vids/day2_3_29/._itinerary_5_2.MP4
Skipping macOS metadata file: ../mumbai_vids/day3_3_31/._itinerary_1.MP4
Skipping macOS metadata file: ../mumbai_vids/day3_3_31/._itinerary_3.MP4
Skipping macOS metadata file: ../mumbai_vids/day3_3_31/._itinerary_2.MP4
Skipping macOS metadata file: ../mumbai_vids/day3_3_31/._itinerary_5.MP4
Skipping macOS metadata file: ../